
# 23 — Preference Data, Sequence Log-Probabilities, and Reward Modeling

## Goal

Understand how language-model post-training can learn from
preferences between complete responses.

We will build preference data representations, calculate sequence
log-probabilities, derive the Bradley-Terry preference objective,
and train a small reward model.

## 01. Preference Data and Pairwise Comparisons

A preference example contains a prompt, a chosen response,
and a rejected response.

Preference is conditional on the prompt. It does not imply
that one response is universally better than another.


In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PreferenceExample:
    """Represent one prompt-conditioned pairwise preference.

    Args:
        prompt: Instruction shared by both responses.
        chosen: Preferred response.
        rejected: Less preferred response.
    """

    prompt: str
    chosen: str
    rejected: str


preferences = [
    PreferenceExample("Say yes", "Yes", "No"),
    PreferenceExample("Say no", "No", "Yes"),
    PreferenceExample("Say hello", "Hello", "Farewell"),
]
for example in preferences:
    print(example)

PreferenceExample(prompt='Say yes', chosen='Yes', rejected='No')
PreferenceExample(prompt='Say no', chosen='No', rejected='Yes')
PreferenceExample(prompt='Say hello', chosen='Hello', rejected='Farewell')


In [5]:
# Validate basic pairwise preference structure.
for example in preferences:
    assert example.prompt
    assert example.chosen
    assert example.rejected
    assert example.chosen != example.rejected

# The same response ranking can reverse under another prompt.
assert preferences[0].chosen == preferences[1].rejected
assert preferences[0].rejected == preferences[1].chosen
assert preferences[0].prompt != preferences[1].prompt

print("Prompt-conditioned preference test passed.")


Prompt-conditioned preference test passed.



### Preference Data — Takeaways

Supervised fine-tuning learns from selected reference responses.

Preference learning uses relative judgments between candidate
responses conditioned on a shared prompt.

A preference pair does not directly provide an absolute reward,
nor does it specify the probability of either response.

Response ranking may reverse when the prompt changes.

The next step is to calculate token-level and complete-response
log-probabilities under a language model.



## 02. Token vs Sequence Log-Probabilities

### Why complete-response probabilities?

A causal language model predicts one token at a time.

For a prompt x and completion y, the probability chain rule gives:

$$
\log \pi_\theta(y \mid x)
=
\sum_{t=0}^{T_y-1}
\log \pi_\theta(y_t \mid x,y_{<t}).
$$

Sequence log-probability is a sum, not a mean.

The following pipeline converts GPT logits into a score for a
complete candidate response.

```mermaid
flowchart TD
    A["Prompt x + Completion y"] --> B["Tokenize full sequence"]
    B --> C["Shift inputs and labels once"]
    C --> D["GPT forward: logits (B, T, V)"]
    D --> E["log_softmax over vocabulary V"]
    E --> F["Gather log p of actual target token"]
    F --> G["Mask prompt and padding positions"]
    G --> H["Sum completion-token log probabilities"]
    H --> I["log π(y | x)"]
```

This is teacher-forced scoring. The model receives the actual
previous response tokens when scoring each target character.

The resulting sequence log-probability is a language-model
likelihood, not a learned preference reward.


In [6]:
import torch

# (T=2,): conditional probabilities of two successive target tokens
token_probabilities = torch.tensor([0.8, 0.9])

# (): product of the conditional token probabilities
sequence_probability = token_probabilities.prod()

# (): sum of the conditional token log-probabilities
sequence_log_probability = token_probabilities.log().sum()

torch.testing.assert_close(sequence_log_probability, sequence_probability.log())

print("Sequence probability:", sequence_probability.item())
print("Sequence log-probability:", sequence_log_probability.item())


Sequence probability: 0.7199999690055847
Sequence log-probability: -0.3285040855407715


In [7]:
from pathlib import Path

from llmfp.data import (
    CharacterTokenizer,
    pad_supervised_batch,
    shift_supervised_tokens,
)
from llmfp.models import GPT, GPTConfig
from llmfp.training import per_example_sft_nll

checkpoint_path = Path("../checkpoints/tiny_shakespeare_sft/step_00040.pt")

if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"Missing {checkpoint_path}. "
        "Complete the checkpoint-saving cell in Lesson 22 first."
    )

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

tokenizer = CharacterTokenizer(checkpoint["tokenizer_vocabulary"])

config = GPTConfig(**checkpoint["model_config"])

sft_policy = GPT(config)

sft_policy.load_state_dict(checkpoint["model_state_dict"], strict=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sft_policy = sft_policy.to(device)
sft_policy.eval()

assert tokenizer.vocab_size == config.vocab_size == 65

print("Loaded SFT checkpoint:", checkpoint_path.name)
print("Vocabulary size:", config.vocab_size)
print("Device:", device)


Loaded SFT checkpoint: step_00040.pt
Vocabulary size: 65
Device: cuda


In [8]:
candidate_examples = []
candidate_metadata = []

for pair_index, example in enumerate(preferences):
    prompt_text = f"User: {example.prompt}\nAssistant: "

    for kind, response in (
        ("chosen", example.chosen),
        ("rejected", example.rejected),
    ):
        completion_text = response + "\n"
        full_text = prompt_text + completion_text

        # (L,): serialized prompt and completion character IDs
        full_ids = torch.tensor(tokenizer.encode(full_text), dtype=torch.long)

        # (L,): full-sequence positions selected for supervision
        target_mask = torch.zeros_like(full_ids, dtype=torch.bool)

        prompt_length = len(tokenizer.encode(prompt_text))
        target_mask[prompt_length:] = True

        # (T=L-1,): causally aligned inputs and completion-only labels
        input_ids, labels = shift_supervised_tokens(full_ids, target_mask)

        # (R,): supervised response character IDs, including newline
        selected_labels = labels[labels.ne(-100)]

        torch.testing.assert_close(
            selected_labels,
            torch.tensor(tokenizer.encode(completion_text), dtype=torch.long),
        )

        candidate_examples.append((input_ids, labels))
        candidate_metadata.append(
            {
                "pair_index": pair_index,
                "kind": kind,
                "prompt": example.prompt,
                "response": response,
                "completion": completion_text,
            }
        )

# Use a valid in-vocabulary character for right padding.
pad_token_id = tokenizer.token_to_id[" "]

# (B=6, T_max): padded predictor inputs
batch_inputs, batch_labels = pad_supervised_batch(
    candidate_examples, pad_token_id=pad_token_id
)

assert batch_inputs.shape == batch_labels.shape
assert batch_inputs.shape[0] == 6
assert batch_labels.ne(-100).sum().item() == 29

print("Candidate batch:", batch_inputs.shape)
print("Total supervised characters:", 29)


Candidate batch: torch.Size([6, 35])
Total supervised characters: 29


In [9]:
import torch.nn.functional as F

# (B=6, T_max): predictor inputs on the model device
inputs_device = batch_inputs.to(device)

# (B=6, T_max): supervised labels on the model device
labels_device = batch_labels.to(device)

with torch.no_grad():
    # (B=6, T_max, V=65): GPT next-token predictions
    logits = sft_policy(inputs_device)

    # (B=6, T_max, V=65): normalized vocabulary log-probabilities
    log_probs = F.log_softmax(logits, dim=-1)

    # (B=6, T_max): supervised completion positions
    completion_mask = labels_device.ne(-100)

    # (B=6, T_max): safe indices for gathering vocabulary values
    safe_labels = labels_device.masked_fill(~completion_mask, 0)

    # (B=6, T_max): log-probability of the actual next token
    selected_log_probs = log_probs.gather(
        dim=-1, index=safe_labels.unsqueeze(-1)
    ).squeeze(-1)

    # (B=6, T_max): zero out prompt and padding contributions
    token_log_probs = selected_log_probs.masked_fill(~completion_mask, 0.0)

    # (B=6,): log-probability of each full completion
    sequence_log_probs = token_log_probs.sum(dim=1)

    # (B=6,): supervised token counts per completion
    completion_lengths = completion_mask.sum(dim=1)

    # (B=6,): mean NLL, using our Lesson 22 reusable primitive
    mean_nll = per_example_sft_nll(logits, labels_device)

# The sequence score must equal minus mean NLL times token count.
torch.testing.assert_close(sequence_log_probs, -mean_nll * completion_lengths)

assert torch.isfinite(sequence_log_probs).all()

print("Sequence log-probabilities:")
for metadata, score in zip(
    candidate_metadata,
    sequence_log_probs.cpu().tolist(),
):
    print(
        f"{metadata['prompt']:12s} "
        f"{metadata['kind']:8s} "
        f"{metadata['response']:10s} "
        f"logp={score:.4f}"
    )

print("Sequence log-probability consistency passed.")


Sequence log-probabilities:
Say yes      chosen   Yes        logp=-0.0007
Say yes      rejected No         logp=-20.4530
Say no       chosen   No         logp=-18.6250
Say no       rejected Yes        logp=-0.0015
Say hello    chosen   Hello      logp=-0.0011
Say hello    rejected Farewell   logp=-10.3109
Sequence log-probability consistency passed.


In [10]:
import plotly.graph_objects as go

fig = go.Figure()

for row, kind in ((0, "chosen"), (1, "rejected")):
    metadata = candidate_metadata[row]

    # (R,): log-probabilities of actual completion characters
    values = token_log_probs[row, completion_mask[row]].cpu().tolist()

    characters = list(metadata["completion"])

    assert len(values) == len(characters)

    fig.add_trace(
        go.Bar(
            x=list(range(1, len(values) + 1)),
            y=values,
            name=f"{kind.title()}: {metadata['response']}",
            customdata=[repr(char) for char in characters],
            hovertemplate=(
                "Character position: %{x}"
                "<br>Character: %{customdata}"
                "<br>Log-probability: %{y:.4f}"
                "<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Token Log-Probabilities — Prompt: Say yes",
    xaxis_title="Completion character position",
    yaxis_title="Log-probability of target character",
    xaxis=dict(dtick=1),
    barmode="group",
    height=420,
)

fig.show()



### Token and Sequence Log-Probabilities — Takeaways

The probability of a completion is the product of its conditional
next-token probabilities.

Taking the logarithm converts that product into a sum.

We restored the SFT GPT from Lesson 22 and teacher-forced both
chosen and rejected responses through the same policy.

`F.log_softmax` converted vocabulary logits into log-probabilities.

`gather` selected the log-probability of each actual target token.
Ignored labels were replaced with valid gather indices and then
masked out of the result.

Summing the completion-token log-probabilities produced one
sequence log-probability per candidate response.

We verified that sequence log-probability equals negative mean
NLL multiplied by the number of supervised completion tokens.

A policy's sequence log-probability is not a preference reward.
The model is not guaranteed to prefer the annotated chosen response.

The next section examines completion masks, sequence lengths,
and the difference between sum and mean normalization.



## 03. Completion Masks and Length Effects

### Three regions in a padded preference batch

Every prediction position belongs to exactly one region:

1. Prompt/context targets: valid positions excluded from scoring.
2. Completion targets: response tokens included in scoring.
3. Right-padding targets: artificial positions excluded from scoring.

Masks are aligned with next-token prediction positions,
not the original unshifted sequence positions.

For a completion y of R tokens:

$$
S_{\mathrm{sum}}(x,y)
=
\sum_{t=1}^{R}
\log \pi_\theta(y_t \mid x,y_{<t})
$$

$$
S_{\mathrm{mean}}(x,y)
=
\frac{S_{\mathrm{sum}}(x,y)}{R}
$$

The sum is the autoregressive log-probability of the specified
completion. The mean is a length-normalized diagnostic.

In this chapter, the final newline is included in the completion.
It is an ordinary character, not a dedicated EOS token.


In [11]:
# (B=6,): original shifted sequence lengths before right padding
sequence_lengths = torch.tensor(
    [inputs.numel() for inputs, _ in candidate_examples],
    dtype=torch.long,
    device=device,
)

# (1, T_max=35): predictor position indices
position_ids = torch.arange(batch_labels.shape[1], device=device).unsqueeze(0)

# (B=6, T_max=35): original, non-padding predictor positions
valid_target_mask = position_ids < sequence_lengths.unsqueeze(1)

# (B=6, T_max=35): supervised completion prediction positions
completion_target_mask = labels_device.ne(-100)

# (B=6, T_max=35): valid predictor positions ignored as prompt/context
prompt_target_mask = valid_target_mask & ~completion_target_mask

# (B=6, T_max=35): positions introduced by right padding
padding_target_mask = ~valid_target_mask

# (B=6, T_max=35): count membership in each of three regions
region_membership = (
    prompt_target_mask.long()
    + completion_target_mask.long()
    + padding_target_mask.long()
)

assert torch.all(region_membership == 1)
assert torch.equal(completion_target_mask, completion_mask)
assert torch.all(labels_device[padding_target_mask] == -100)
assert completion_target_mask.sum().item() == 29

print("Prompt positions:", prompt_target_mask.sum().item())
print("Completion positions:", completion_target_mask.sum().item())
print("Padding positions:", padding_target_mask.sum().item())


Prompt positions: 146
Completion positions: 29
Padding positions: 35


In [ ]:
import plotly.graph_objects as go

# (B=6, T_max=35): 0=prompt, 1=completion, 2=padding
region_codes = torch.where(
    padding_target_mask, 2, torch.where(completion_target_mask, 1, 0)
)

region_names = {
    0: "Prompt/context — ignored",
    1: "Completion — scored",
    2: "Right padding — ignored",
}

region_text = [
    [region_names[value] for value in row]
    for row in region_codes.cpu().tolist()
]

candidate_labels = [
    f"{meta['prompt']} | {meta['kind']}: {meta['response']}"
    for meta in candidate_metadata
]

mask_fig = go.Figure(
    data=go.Heatmap(
        z=region_codes.cpu().tolist(),
        x=list(range(batch_labels.shape[1])),
        y=candidate_labels,
        text=region_text,
        hovertemplate=(
            "Candidate: %{y}"
            "<br>Prediction position: %{x}"
            "<br>Region: %{text}"
            "<extra></extra>"
        ),
        colorscale=[
            [0.00, "#94A3B8"],
            [0.2499, "#94A3B8"],
            [0.25, "#2563EB"],
            [0.7499, "#2563EB"],
            [0.75, "#E2E8F0"],
            [1.00, "#E2E8F0"],
        ],
        zmin=0,
        zmax=2,
        showscale=False,
        xgap=1,
        ygap=3,
    )
)

mask_fig.update_layout(
    title="Preference Batch — Prompt, Completion, and Padding",
    xaxis_title="Shifted prediction position",
    yaxis=dict(autorange="reversed"),
    height=420,
)

mask_fig.show()


In [ ]:
# (R_chosen=4,): conditional probabilities of chosen tokens
toy_chosen_probs = torch.full((4,), 0.90, dtype=torch.float64)

# (R_rejected=2,): conditional probabilities of rejected tokens
toy_rejected_probs = torch.full((2,), 0.85, dtype=torch.float64)

# (): sum of token log-probabilities for the chosen sequence
toy_chosen_sum = toy_chosen_probs.log().sum()

# (): sum of token log-probabilities for the rejected sequence
toy_rejected_sum = toy_rejected_probs.log().sum()

# (): mean token log-probabilities
toy_chosen_mean = toy_chosen_probs.log().mean()
toy_rejected_mean = toy_rejected_probs.log().mean()

print("Chosen sum:", toy_chosen_sum.item())
print("Rejected sum:", toy_rejected_sum.item())
print("Chosen mean:", toy_chosen_mean.item())
print("Rejected mean:", toy_rejected_mean.item())

# The shorter rejected completion has greater sequence probability.
assert toy_rejected_sum > toy_chosen_sum

# The chosen completion has better average per-token probability.
assert toy_chosen_mean > toy_rejected_mean

print("Length-dependent ranking reversal demonstrated.")


Chosen sum: -0.4214420626313051
Rejected sum: -0.3250378589955499
Chosen mean: -0.10536051565782628
Rejected mean: -0.16251892949777494
Length-dependent ranking reversal demonstrated.


In [14]:
# (B=6,): average log-probability per supervised completion token
average_log_probs = sequence_log_probs / completion_lengths

# Mean log-probability must equal negative per-example mean NLL.
torch.testing.assert_close(average_log_probs, -mean_nll)

for meta, length, total, average in zip(
    candidate_metadata,
    completion_lengths.cpu().tolist(),
    sequence_log_probs.cpu().tolist(),
    average_log_probs.cpu().tolist(),
):
    print(
        f"{meta['prompt']:10s} | "
        f"{meta['kind']:8s} | "
        f"length={length:2d} | "
        f"sum={total:9.4f} | "
        f"mean={average:9.4f}"
    )


Say yes    | chosen   | length= 4 | sum=  -0.0007 | mean=  -0.0002
Say yes    | rejected | length= 3 | sum= -20.4530 | mean=  -6.8177
Say no     | chosen   | length= 3 | sum= -18.6250 | mean=  -6.2083
Say no     | rejected | length= 4 | sum=  -0.0015 | mean=  -0.0004
Say hello  | chosen   | length= 6 | sum=  -0.0011 | mean=  -0.0002
Say hello  | rejected | length= 9 | sum= -10.3109 | mean=  -1.1457


In [15]:
from plotly.subplots import make_subplots

score_labels = [
    f"{meta['prompt']} | {meta['kind']}: {meta['response']}"
    for meta in candidate_metadata
]

length_values = completion_lengths.cpu().tolist()

score_fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    subplot_titles=(
        "Complete-response log P (sum)",
        "Average log P per character",
    ),
    horizontal_spacing=0.08,
)

for column, values in (
    (1, sequence_log_probs.cpu().tolist()),
    (2, average_log_probs.cpu().tolist()),
):
    score_fig.add_trace(
        go.Bar(
            x=values,
            y=score_labels,
            orientation="h",
            customdata=length_values,
            hovertemplate=(
                "Candidate: %{y}"
                "<br>Score: %{x:.4f}"
                "<br>Completion length: %{customdata}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=column,
    )

score_fig.update_xaxes(title_text="Sum of log-probabilities", row=1, col=1)

score_fig.update_xaxes(title_text="Mean log-probability", row=1, col=2)

score_fig.update_yaxes(autorange="reversed", row=1, col=1)

score_fig.update_layout(
    title="Preference Candidates — Two Ways to Aggregate Token Scores",
    height=480,
    width=1100,
    showlegend=False,
)

score_fig.show()


In [16]:
# (B=6, T_max=35, V=65): copied logits for a controlled perturbation
perturbed_logits = logits.detach().clone()

# Modify only ignored predictor positions, across all vocabulary classes.
perturbed_logits[~completion_target_mask] = 50.0

# (B=6, T_max=35, V=65): normalized perturbed log-probabilities
perturbed_log_probs = F.log_softmax(perturbed_logits, dim=-1)

# (B=6, T_max=35): selected next-token log-probabilities
perturbed_selected = perturbed_log_probs.gather(
    dim=-1, index=safe_labels.unsqueeze(-1)
).squeeze(-1)

# (B=6, T_max=35): only completion positions contribute
perturbed_selected = perturbed_selected.masked_fill(
    ~completion_target_mask, 0.0
)

# (B=6,): sequence scores after ignored-position perturbation
perturbed_sequence_scores = perturbed_selected.sum(dim=1)

torch.testing.assert_close(perturbed_sequence_scores, sequence_log_probs)

print("Ignored-position logit invariance passed.")


Ignored-position logit invariance passed.



### Completion Masks and Length Effects — Takeaways

A padded preference batch contains three disjoint target regions:
prompt/context, supervised completion, and right padding.

Only completion targets contribute to sequence log-probability.

Right-padding positions and ignored prompt targets must not
contribute to response scoring.

Sequence log-probability is the sum of completion-token
log-probabilities. Mean log-probability divides this sum by
the number of supervised completion tokens.

Sum and mean normalization can rank candidate completions
differently when their lengths differ.

We demonstrated such a ranking reversal with a controlled
probability example and compared both aggregations using
our fine-tuned GPT.

Length-normalized log-probability is a diagnostic, not the
autoregressive log-probability of the complete response.
It should not silently replace sequence log-probability
inside a later preference-optimization objective.

Policy likelihood is also distinct from an explicitly learned
preference reward.

The next section derives the Bradley-Terry preference model.



## 04. Bradley–Terry Preference Model

### From scalar rewards to preference probabilities

A reward model assigns one scalar to a prompt-response pair:

$$
r_\phi(x,y)\in\mathbb R
$$

The Bradley-Terry model assumes:

$$
P_\phi(y^+\succ y^-\mid x)
=
\frac{\exp(r^+)}{\exp(r^+)+\exp(r^-)}
=
\sigma(r^+-r^-)
$$

where both responses are evaluated under the same prompt
and by the same reward model.

The pairwise negative log-likelihood is:

$$
\mathcal L_{\mathrm{BT}}(\phi)
=
-\frac{1}{B}\sum_{i=1}^B
\log\sigma(r_i^+-r_i^-)
$$

Equivalently:

$$
\mathcal L_{\mathrm{BT}}(\phi)
=
\frac{1}{B}\sum_{i=1}^B
\operatorname{softplus}(-(r_i^+-r_i^-))
$$

```mermaid
flowchart TD
    P["Shared prompt x"] --> A["Reward model: r_phi(x, chosen)"]
    P --> B["Reward model: r_phi(x, rejected)"]
    C["Chosen response"] --> A
    D["Rejected response"] --> B
    A --> E["Reward margin: r+ - r-"]
    B --> E
    E --> F["Sigmoid: preference probability"]
    F --> G["Negative log-likelihood"]
```

These are reward-model scores, not language-model
sequence log-probabilities.

The scores in this subsection are manually constructed.
We do not train the reward model until Section 05.


In [17]:
import torch
import torch.nn.functional as F

# (): manually assigned reward for the chosen response
r_chosen = torch.tensor(2.0)

# (): manually assigned reward for the rejected response
r_rejected = torch.tensor(0.0)

# (): reward margin between the two responses
reward_margin = r_chosen - r_rejected

# (): Bradley-Terry probability from exponential strengths
bt_probability_exp = r_chosen.exp() / (r_chosen.exp() + r_rejected.exp())

# (): equivalent probability using the sigmoid function
bt_probability = torch.sigmoid(reward_margin)

torch.testing.assert_close(bt_probability_exp, bt_probability)

# (): direct negative log-likelihood
bt_loss_direct = -bt_probability.log()

# (): numerically stable equivalent using softplus
bt_loss_stable = F.softplus(-reward_margin)

torch.testing.assert_close(bt_loss_direct, bt_loss_stable)

print("Reward margin:", reward_margin.item())
print("Preference probability:", bt_probability.item())
print("Pairwise loss:", bt_loss_stable.item())


Reward margin: 2.0
Preference probability: 0.8807970285415649
Pairwise loss: 0.12692800164222717


In [18]:
# (B=3,): synthetic reward scores for chosen responses
chosen_rewards = torch.tensor([1.0, -1.0, 0.5], requires_grad=True)

# (B=3,): synthetic reward scores for rejected responses
rejected_rewards = torch.tensor([0.0, 1.0, 0.5], requires_grad=True)

# (B=3,): per-pair chosen-minus-rejected reward margins
bt_margins = chosen_rewards - rejected_rewards

# (B=3,): probability assigned to each observed preference
bt_probabilities = torch.sigmoid(bt_margins)

# (B=3,): negative log-likelihood for each preference pair
bt_pair_losses = F.softplus(-bt_margins)

# (): average pairwise negative log-likelihood
bt_mean_loss = bt_pair_losses.mean()

# Verify the equivalent log-sigmoid formulation.
torch.testing.assert_close(bt_mean_loss, -F.logsigmoid(bt_margins).mean())

assert len(preferences) == len(bt_margins) == 3

for example, margin, probability, loss_value in zip(
    preferences,
    bt_margins.detach().tolist(),
    bt_probabilities.detach().tolist(),
    bt_pair_losses.detach().tolist(),
):
    print(
        f"{example.prompt:12s} | "
        f"margin={margin:+.2f} | "
        f"P(chosen)={probability:.4f} | "
        f"loss={loss_value:.4f}"
    )

print("Mean BT loss:", bt_mean_loss.item())


Say yes      | margin=+1.00 | P(chosen)=0.7311 | loss=0.3133
Say no       | margin=-2.00 | P(chosen)=0.1192 | loss=2.1269
Say hello    | margin=+0.00 | P(chosen)=0.5000 | loss=0.6931
Mean BT loss: 1.0444456338882446


In [19]:
# (B=3,): gradients with respect to synthetic chosen rewards
# (B=3,): gradients with respect to synthetic rejected rewards
grad_chosen, grad_rejected = torch.autograd.grad(
    bt_mean_loss, (chosen_rewards, rejected_rewards)
)

batch_size = len(preferences)

# (B=3,): analytical gradients, including batch-mean scaling
expected_chosen_grad = (bt_probabilities.detach() - 1.0) / batch_size

# (B=3,): analytical gradients for rejected rewards
expected_rejected_grad = (1.0 - bt_probabilities.detach()) / batch_size

torch.testing.assert_close(grad_chosen, expected_chosen_grad)

torch.testing.assert_close(grad_rejected, expected_rejected_grad)

assert torch.all(grad_chosen < 0)
assert torch.all(grad_rejected > 0)

torch.testing.assert_close(
    grad_chosen + grad_rejected, torch.zeros_like(grad_chosen)
)

print("Chosen reward gradients:", grad_chosen.tolist())
print("Rejected reward gradients:", grad_rejected.tolist())
print("BT analytical gradient checks passed.")


Chosen reward gradients: [-0.08964714407920837, -0.29359903931617737, -0.1666666716337204]
Rejected reward gradients: [0.08964714407920837, 0.29359903931617737, 0.1666666716337204]
BT analytical gradient checks passed.


In [20]:
# (B=3,): common offsets applied to both rewards in each pair
common_offsets = torch.tensor([5.0, -10.0, 2.0])

with torch.no_grad():
    # (B=3,): BT probabilities after adding common offsets
    shifted_probabilities = torch.sigmoid(
        (chosen_rewards + common_offsets) - (rejected_rewards + common_offsets)
    )

    # (B=3,): probability after swapping the candidate order
    swapped_probabilities = torch.sigmoid(rejected_rewards - chosen_rewards)

torch.testing.assert_close(shifted_probabilities, bt_probabilities.detach())

torch.testing.assert_close(
    swapped_probabilities, 1.0 - bt_probabilities.detach()
)

print("Common reward-shift invariance passed.")
print("Swapped-response probability symmetry passed.")


Common reward-shift invariance passed.
Swapped-response probability symmetry passed.


In [21]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# (N=241,): candidate reward differences from -6 to +6
margin_grid = torch.linspace(-6.0, 6.0, steps=241)

# (N=241,): Bradley-Terry preference probabilities
probability_curve = torch.sigmoid(margin_grid)

# (N=241,): negative log-likelihood for observed chosen preferences
loss_curve = F.softplus(-margin_grid)

bt_fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "P(Chosen preferred)",
        "Pairwise negative log-likelihood",
    ),
    horizontal_spacing=0.12,
)

bt_fig.add_trace(
    go.Scatter(
        x=margin_grid.tolist(),
        y=probability_curve.tolist(),
        mode="lines",
        name="Preference probability",
    ),
    row=1,
    col=1,
)

bt_fig.add_trace(
    go.Scatter(
        x=margin_grid.tolist(),
        y=loss_curve.tolist(),
        mode="lines",
        name="Pairwise loss",
    ),
    row=1,
    col=2,
)

bt_fig.add_trace(
    go.Scatter(
        x=bt_margins.detach().tolist(),
        y=bt_probabilities.detach().tolist(),
        mode="markers",
        name="Toy preference pairs",
        text=[example.prompt for example in preferences],
        hovertemplate=(
            "Prompt: %{text}"
            "<br>Reward margin: %{x:.2f}"
            "<br>Preference probability: %{y:.4f}"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

bt_fig.update_xaxes(title_text="Reward margin (chosen - rejected)")

bt_fig.update_yaxes(title_text="Probability", range=[0, 1], row=1, col=1)

bt_fig.update_yaxes(title_text="Loss", row=1, col=2)

bt_fig.update_layout(
    title="Bradley-Terry Preference Probability and Loss",
    height=440,
    width=1050,
)

bt_fig.show()



### Bradley–Terry Preference Model — Takeaways

A reward model assigns one scalar score to a prompt-response pair.

The Bradley-Terry model converts a chosen-minus-rejected reward
difference into a pairwise preference probability using sigmoid.

Pairwise negative log-likelihood can be implemented stably as
`F.softplus(-reward_margin)`.

The analytical gradients with respect to independent reward scores
push chosen rewards upward and rejected rewards downward under
gradient descent.

The loss depends on relative reward differences, so adding the
same constant to both scores leaves preference probabilities
unchanged.

Reward scale still matters because scaling the difference changes
the preference probability.

Our reward values were manually constructed, not learned.
The BT probability is a modeling assumption and is not guaranteed
to be calibrated to human preference frequencies.

Policy sequence log-probabilities and learned reward scores
must remain conceptually distinct.

The next step is to implement and train a small reward model.



## 05. Training a Reward Model

### From a GPT hidden state to a scalar reward

We reuse the representations learned by the SFT GPT, but replace
next-token prediction with a sequence-level scalar score.

For this first experiment, we freeze the pretrained backbone
and train only a linear reward head:

$$
r_w(x,y) = w^\top h_{\theta_{\mathrm{SFT}}}(x,y)
$$

The representation is taken from the final real token position
of the serialized prompt-response sequence.

The backbone parameters remain fixed. Only w is optimized.

```mermaid
flowchart TD
    A["Prompt + chosen response"] --> C["Shared frozen GPT backbone"]
    B["Prompt + rejected response"] --> C
    C --> D["Final real-token hidden states"]
    D --> E["Shared trainable scalar reward head"]
    E --> F["Chosen and rejected rewards"]
    F --> G["Reward difference"]
    G --> H["Bradley-Terry loss"]
    H --> I["Update reward-head parameters"]
```

This is a frozen-backbone reward probe, not full-parameter
reward-model fine-tuning.

The preference labels are synthetic teaching examples,
not collected human preference annotations.


In [ ]:
reward_sequences = []

for metadata in candidate_metadata:
    full_text = (
        f"User: {metadata['prompt']}\nAssistant: {metadata['completion']}"
    )

    # (L_i,): complete prompt-response sequence including final newline
    sequence = torch.tensor(tokenizer.encode(full_text), dtype=torch.long)

    reward_sequences.append(sequence)

batch_size = len(reward_sequences)
max_reward_length = max(sequence.numel() for sequence in reward_sequences)

# (B=6,): original sequence lengths, including final newline
reward_lengths = torch.tensor(
    [sequence.numel() for sequence in reward_sequences], dtype=torch.long
)

# (B=6, T_max): right-padded complete sequences
reward_inputs = torch.full(
    (batch_size, max_reward_length), pad_token_id, dtype=torch.long
)

for row, sequence in enumerate(reward_sequences):
    reward_inputs[row, : sequence.numel()] = sequence

# (B=6,): position of the last real character in each sequence
last_token_positions = reward_lengths - 1

# (B=6,): final real token ID for each sequence, not the padding ones
last_token_ids = reward_inputs[torch.arange(batch_size), last_token_positions]

assert batch_size == 2 * len(preferences) == 6
assert torch.all(last_token_ids == tokenizer.token_to_id["\n"])

for index in range(len(preferences)):
    assert candidate_metadata[2 * index]["kind"] == "chosen"
    assert candidate_metadata[2 * index + 1]["kind"] == "rejected"

print("Reward input shape:", reward_inputs.shape)
print("Last real-token positions:", last_token_positions.tolist())


Reward input shape: torch.Size([6, 36])
Last real-token positions: [28, 27, 26, 27, 32, 35]


In [ ]:
import copy

import torch.nn as nn


class FrozenGPTRewardModel(nn.Module):
    """Score complete sequences with frozen GPT features and a learned head.

    Args:
        policy: Existing GPT whose representations initialize the backbone.
    """

    def __init__(self, policy: GPT) -> None:
        super().__init__()

        self.token_embedding = copy.deepcopy(policy.token_embedding)
        self.blocks = copy.deepcopy(policy.blocks)
        self.final_norm = copy.deepcopy(policy.final_norm)

        # Freeze all pretrained representation parameters.
        self.token_embedding.requires_grad_(False)
        self.blocks.requires_grad_(False)
        self.final_norm.requires_grad_(False)

        # No bias: a shared scalar bias cancels in every BT reward margin.
        self.reward_head = nn.Linear(policy.config.embedding_dim, 1, bias=False)

        # Initial rewards are all zero, giving BT probability 0.5.
        nn.init.zeros_(self.reward_head.weight)

    @torch.no_grad()
    def encode(
        self,
        token_ids: torch.Tensor,
        last_positions: torch.Tensor,
    ) -> torch.Tensor:
        """Return final real-token hidden states from the frozen GPT.

        Args:
            token_ids: (B, T) complete, right-padded token IDs.
            last_positions: (B,) last real-token indices.

        Returns:
            (B, C) final real-token hidden representations.
        """
        # (B, T, C): input token embeddings
        hidden = self.token_embedding(token_ids)

        for block in self.blocks:
            block_output = block(hidden)

            if not isinstance(block_output, torch.Tensor):
                raise RuntimeError("Expected uncached Transformer output.")

            # (B, T, C): updated residual-stream representations
            hidden = block_output

        # (B, T, C): final normalized hidden states
        hidden = self.final_norm(hidden)

        # (B,): sequence indices used to select one state per row
        rows = torch.arange(token_ids.shape[0], device=token_ids.device)

        # (B, C): the hidden state of each sequence's final real token
        final_features = hidden[rows, last_positions]

        return final_features

    def forward(
        self,
        token_ids: torch.Tensor,
        last_positions: torch.Tensor,
    ) -> torch.Tensor:
        """Return one differentiable scalar reward per sequence."""
        # (B, C): frozen contextual sequence representations
        features = self.encode(token_ids, last_positions)

        # (B,): scalar reward scores
        rewards = self.reward_head(features).squeeze(-1)

        return rewards


In [24]:
reward_model = FrozenGPTRewardModel(sft_policy).to(device)
reward_model.eval()

# (B=6, T_max): complete sequences on the model device
reward_inputs_device = reward_inputs.to(device)

# (B=6,): final real-token indices on the model device
last_positions_device = last_token_positions.to(device)

with torch.no_grad():
    # (B=6, C=256): frozen final-token representations
    frozen_features = reward_model.encode(
        reward_inputs_device, last_positions_device
    )

    # (B=6, T_max, V=65): original GPT vocabulary logits
    reference_logits = sft_policy(reward_inputs_device)

    # (B=6,): row indices
    row_indices = torch.arange(batch_size, device=device)

    # (B=6, V=65): original GPT logits at final real-token positions
    reference_last_logits = reference_logits[row_indices, last_positions_device]

    # (B=6, V=65): recover LM logits from copied hidden states
    reconstructed_logits = F.linear(frozen_features, sft_policy.lm_head.weight)

    # (B=6,): initial scalar rewards
    initial_rewards = reward_model(reward_inputs_device, last_positions_device)

torch.testing.assert_close(
    reconstructed_logits, reference_last_logits, rtol=1e-4, atol=1e-5
)

torch.testing.assert_close(initial_rewards, torch.zeros_like(initial_rewards))

# (B=3,): initial reward margins
initial_margins = initial_rewards[0::2] - initial_rewards[1::2]

# (): initial mean BT loss
initial_reward_loss = F.softplus(-initial_margins).mean()

torch.testing.assert_close(
    initial_reward_loss,
    torch.tensor(2.0, device=device).log(),
)

print("Frozen feature shape:", frozen_features.shape)
print("Initial reward loss:", initial_reward_loss.item())
print("GPT feature reconstruction passed.")


Frozen feature shape: torch.Size([6, 256])
Initial reward loss: 0.6931471824645996
GPT feature reconstruction passed.


In [25]:
reward_learning_rate = 1e-2
reward_training_steps = 80

reward_optimizer = torch.optim.AdamW(
    reward_model.reward_head.parameters(),
    lr=reward_learning_rate,
    weight_decay=0.0,
)

# (1, C=256): original reward-head weights
initial_head_weights = reward_model.reward_head.weight.detach().clone()

reward_loss_history = [initial_reward_loss.item()]

for step in range(1, reward_training_steps + 1):
    reward_optimizer.zero_grad(set_to_none=True)

    # (B=6,): scores produced by the shared trainable reward head
    reward_scores = reward_model.reward_head(frozen_features).squeeze(-1)

    # (B_pairs=3,): chosen response scores
    chosen_scores = reward_scores[0::2]

    # (B_pairs=3,): rejected response scores
    rejected_scores = reward_scores[1::2]

    # (B_pairs=3,): preference margins
    margins = chosen_scores - rejected_scores

    # (): Bradley-Terry mean negative log-likelihood
    loss = F.softplus(-margins).mean()

    loss.backward()

    if step == 1:
        head_gradient = reward_model.reward_head.weight.grad

        assert head_gradient is not None
        assert torch.isfinite(head_gradient).all()
        assert head_gradient.norm().item() > 0

    reward_optimizer.step()

    with torch.no_grad():
        # (B=6,): updated reward scores after the optimizer step
        updated_scores = reward_model.reward_head(frozen_features).squeeze(-1)

        # (B_pairs=3,): updated pairwise reward margins
        updated_margins = updated_scores[0::2] - updated_scores[1::2]

        # (): updated BT loss
        updated_loss = F.softplus(-updated_margins).mean()

    reward_loss_history.append(updated_loss.item())

    if step in (1, 20, 40, 80):
        print(f"Step {step:02d} | BT loss {updated_loss.item():.4f}")


Step 01 | BT loss 0.5486
Step 20 | BT loss 0.1490
Step 40 | BT loss 0.0507
Step 80 | BT loss 0.0191


In [26]:
reward_model.eval()

with torch.no_grad():
    # (B=6,): trained scalar rewards from full input sequences
    trained_rewards = reward_model(reward_inputs_device, last_positions_device)

    # (B=6,): scores computed from the cached representations
    cached_rewards = reward_model.reward_head(frozen_features).squeeze(-1)

    # (B_pairs=3,): chosen-minus-rejected reward differences
    final_reward_margins = trained_rewards[0::2] - trained_rewards[1::2]

    # (B_pairs=3,): predicted preference probabilities
    final_preference_probs = torch.sigmoid(final_reward_margins)

    # (): final Bradley-Terry training loss
    final_reward_loss = F.softplus(-final_reward_margins).mean()

    # (): reward-head parameter-change norm
    reward_head_change = (
        reward_model.reward_head.weight - initial_head_weights
    ).norm()

torch.testing.assert_close(trained_rewards, cached_rewards)

assert final_reward_loss < initial_reward_loss
assert reward_head_change.item() > 0
assert torch.isfinite(final_preference_probs).all()

# Only the scalar reward head should have trainable parameters.
assert all(
    not parameter.requires_grad
    for module in (
        reward_model.token_embedding,
        reward_model.blocks,
        reward_model.final_norm,
    )
    for parameter in module.parameters()
)

for index, example in enumerate(preferences):
    print(
        f"{example.prompt:12s} | "
        f"margin={final_reward_margins[index].item():+.4f} | "
        f"P(chosen)={final_preference_probs[index].item():.4f}"
    )

print("Initial BT loss:", initial_reward_loss.item())
print("Final BT loss:", final_reward_loss.item())
print("Reward-head weight change:", reward_head_change.item())
print("Frozen-backbone reward training checks passed.")


Say yes      | margin=+3.5286 | P(chosen)=0.9715
Say no       | margin=+3.6016 | P(chosen)=0.9734
Say hello    | margin=+6.5011 | P(chosen)=0.9985
Initial BT loss: 0.6931471824645996
Final BT loss: 0.01911216601729393
Reward-head weight change: 4.193849563598633
Frozen-backbone reward training checks passed.


In [27]:
import plotly.graph_objects as go

reward_fig = go.Figure()

reward_fig.add_trace(
    go.Scatter(
        x=list(range(len(reward_loss_history))),
        y=reward_loss_history,
        mode="lines",
        name="Bradley-Terry training loss",
        hovertemplate=(
            "Optimizer step: %{x}<br>BT loss: %{y:.5f}<extra></extra>"
        ),
    )
)

reward_fig.add_hline(
    y=float(initial_reward_loss.item()),
    line_dash="dash",
    annotation_text="Initial chance-level loss",
)

reward_fig.update_layout(
    title="Training a Frozen-Backbone GPT Reward Head",
    xaxis_title="Optimizer step",
    yaxis_title="Mean Bradley-Terry loss",
    height=420,
)

reward_fig.show()



### Training a Reward Model — Takeaways

We built a minimal reward model using the representations of our
previously fine-tuned GPT.

The model extracts the hidden state at the final real token
position of each complete prompt-response sequence.

A new scalar reward head maps these contextual representations
to one score per response.

We froze the GPT backbone and trained only the reward head.

The reward-head bias was omitted because a shared additive bias
cancels from Bradley-Terry reward differences.

Zero initialization gave equal initial rewards and a
chance-level BT loss of log(2).

Pairwise preferences supplied differentiable supervision
through the Bradley-Terry objective.

We measured training loss, reward margins, preference
probabilities, and parameter changes.

This experiment demonstrates frozen-backbone reward-head
training, not full reward-model fine-tuning.

Low training loss does not establish generalization to
unseen prompt-response pairs.

The next section evaluates reward ranking and generalization.
